# Homework 2: BERT on AWS

In this homework, we will apply the BERT algorithm on Amazon Web Services.

This DataRec repository contains a pointer to accessing recommendation system data, installable via

`pip install datarec-lib`

### Task 1: Download the MovieLens 1m dataset. You should output a copy of the dataset on an AWS S3 bucket.

First, I set up aws CLI and ran `aws configure sso` with the Northwestern start URL and `us-east-2`.
aws s3 ls --profile mse-tl-dataeng300-EMR-549787090008

In [21]:
# Download Starfield Digital Certificate
!curl -O https://www.amazontrust.com/repository/AmazonRootCA1.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA2.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA3.pem
!curl -O https://www.amazontrust.com/repository/AmazonRootCA4.pem
!curl -O https://certs.secureserver.net/repository/sf-class2-root.crt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1188  100  1188    0     0  10555      0 --:--:-- --:--:-- --:--:-- 10607
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1883  100  1883    0     0  12380      0 --:--:-- --:--:-- --:--:-- 12388
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   656  100   656    0     0   6893      0 --:--:-- --:--:-- --:--:--  6905
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   737  100   737    0     0   7315      0 --:--:-- --:--:-- --:--:--  7370
  % Total    % Received % Xferd  Average Speed   Tim

In [27]:
# combine key files
!cat AmazonRootCA1.pem AmazonRootCA2.pem AmazonRootCA3.pem AmazonRootCA4.pem sf-class2-root.crt > keyspaces-bundle.pem


In [15]:
!pip install cassandra-sigv4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 7.6 MB/s eta 0:00:00a 0:00:01


In [28]:
# connection to AWS
from cassandra.cluster import Cluster
from ssl import SSLContext, PROTOCOL_TLSv1_2 , CERT_REQUIRED
import boto3
from cassandra_sigv4.auth import SigV4AuthProvider

ssl_context = SSLContext(PROTOCOL_TLSv1_2)
ssl_context.load_verify_locations('./keyspaces-bundle.pem')
ssl_context.verify_mode = CERT_REQUIRED

/var/folders/82/knv6ptsj1ysgdp6n2mpwwn5m0000gn/T/ipykernel_58128/4213881604.py:7: DeprecationWarning: ssl.PROTOCOL_TLSv1_2 is deprecated
  ssl_context = SSLContext(PROTOCOL_TLSv1_2)


In [30]:
import pandas as pd
access_key = pd.read_csv('../admin/de300-keyspaces_accessKeys.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../admin/de300-keyspaces_accessKeys.csv'

### Task 2. Using only movies that are released before the year 1980, inclusive. Create embeddings for the BERT algorithm. For the same set of items (movies) in the dataset, you should create the embeddings once and output a copy of the necessary intermediate results on the S3 bucket.

In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
@with torch_no_grad
def bert_embed(tokenizer, encoder, texts, max_len=1024):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)

def embed_to_vec():
    items["text"] = items["title"] + ". " + items["description"]
    item_vecs = bert_embed(items["text"].tolist())
    item_id_list = items["item_id"].tolist()

    # Build ANN index (inner product works with normalized vectors)
    index = faiss.IndexFlatIP(item_vecs.shape[1])
    index.add(item_vecs)